# Quick Colab smoke test for NetCDF + LaMa integration
This notebook shows how to run a minimal smoke test in Colab (or locally) using the repository configs and the importable `run(cfg)` entrypoint.

Steps:
1. Install requirements (run once).
2. Compose Hydra config pointing to a small/synthetic NetCDF or a test file.
3. Import `run` from `bin.run` and execute a quick forward/backward pass.

In [ ]:
# Install dependencies (Colab) — adjust if running locally.
!pip install -q xarray netcdf4 hydra-core omegaconf pytorch-lightning
# For Colab it's recommended to install torch with the appropriate wheel; the line below is an example:
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# Show repository files (ensure notebook is at repo root in Colab)
import os
print('CWD', os.getcwd())
print('Listing top-level:')
print('
'.join(sorted(os.listdir('.'))))

In [ ]:
# Compose Hydra config and run smoke test
from hydra import initialize, compose
from omegaconf import OmegaConf

# Adjust overrides to point to a small NetCDF file available in Colab or a synthetic path
with initialize(config_path="configs", job_name="colab-smoke"):
    cfg = compose(config_name="training/lama_small_netcdf.yaml", overrides=[
        "data.dataset.nc_path=/content/data/test.nc",
        "data.dataset.var_names=['temperature','salinity']",
        "training.batch_size=1",
        "training.num_workers=0",
        "training.max_epochs=1",
    ])

print(OmegaConf.to_yaml(cfg))

# Import and run the smoke function
from bin.run import run
res = run(cfg, smoke=True)
print('Smoke result:', res)

Notes:
- If you don't have a real NetCDF file, create a tiny synthetic NetCDF with `xarray` or update `data.dataset.nc_path` to a test file uploaded to Colab.
- Use `training.num_workers=0` if you encounter file access issues.
- The `bin/run.py` entrypoint is import-safe (does not execute on import) and designed for programmatic testing.